# Pore-Water-viz の準備（初回と更新）

GitHub のコードを `Colab Notebooks/Pore-Water-viz` にコピーします。

`yaml/`、`data/`、`output/` は、すでにフォルダがあっても消しません。`yaml/base.yaml` が無いときだけ、`templates/yaml` から作ります。

In [ ]:
import os
import shutil
import urllib.request
import zipfile

BASE_DIR = "/content/drive/MyDrive/Colab Notebooks"
REPO_NAME = "Pore-Water-viz"
ZIP_URL = "https://codeload.github.com/HirotoshiMori/Pore-Water-viz/zip/refs/heads/main"
LOCAL_SRC = "/content/Pore-Water-viz-main"
ZIP_PATH = "/content/Pore-Water-viz-main.zip"
KEEP = {"yaml", "data", "output"}

repo_path = os.path.join(BASE_DIR, REPO_NAME)

from google.colab import drive
drive.mount("/content/drive")

print("Downloading repository...")
urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
if os.path.isdir(LOCAL_SRC):
    shutil.rmtree(LOCAL_SRC)
with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall("/content")
if not os.path.isdir(LOCAL_SRC):
    raise FileNotFoundError("展開後のフォルダが見つかりません: " + LOCAL_SRC)

os.makedirs(repo_path, exist_ok=True)
for name in sorted(os.listdir(LOCAL_SRC)):
    if name in KEEP:
        print("  kept:", name)
        continue
    src = os.path.join(LOCAL_SRC, name)
    dst = os.path.join(repo_path, name)
    if os.path.isdir(src):
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        elif os.path.exists(dst):
            os.remove(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print("  updated:", name)

yaml_dir = os.path.join(repo_path, "yaml")
template = os.path.join(repo_path, "templates", "yaml")
base_yaml = os.path.join(yaml_dir, "base.yaml")
if not os.path.isfile(base_yaml) and os.path.isdir(template):
    os.makedirs(yaml_dir, exist_ok=True)
    for dirpath, _, filenames in os.walk(template):
        rel = os.path.relpath(dirpath, template)
        target_dir = yaml_dir if rel == "." else os.path.join(yaml_dir, rel)
        os.makedirs(target_dir, exist_ok=True)
        for filename in filenames:
            target = os.path.join(target_dir, filename)
            if os.path.exists(target):
                continue
            shutil.copy2(os.path.join(dirpath, filename), target)
            print("  created:", os.path.relpath(target, repo_path))

print("準備が完了しました。")
print(repo_path)
print("yaml/ と data/ と output/ は、この処理では消しません。")
